## Test Grid SANOS



In [8]:
%load_ext autoreload
%autoreload 1

from quantdata.dysanos.simple.aec import load_data
data = load_data(["SPX"]).data['SPX']
print(f"/done; columns {data.keys()}")



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
00: Generating samples for 1 tickers SPX with from 2015-01-01 to 2025-07-01... done; this took 2s.
00: Found the following number of samples per ticker: SPX: 2617.
00: Found date range 2015-01-02 to 2025-07-01 with 2617 total dates. First validation date for 25 steps per sample is 2023-05-01.
00: Using up to 1000 options per date.
01:   Ticker SPX: using 411 training samples (accross 2079 training dates), and 103 validation samples (accross 538 validation dates).
00: Created 411 training samples and 103 validation samples based on the date split at 2023-05-01. That implies 20% of all samples are validation samples.
00: Total time taken: 2s.
/done; columns dict_keys(['dates', 'ask_pure_delta', 'ask_pure_dk', 'ask_pure_gamma', 'ask_pure_price', 'ask_pure_theta', 'ask_pure_vega', 'ask_vol', 'bid_pure_delta', 'bid_pure_dk', 'bid_pure_gamma', 'bid_pure_price', 'bid_pure_theta', 'bid_pure_vega', 'bid_vol'

In [9]:
%aimport quantdata.utils.arbfree

from cdxcore.bs import bs
from cdxcore.dynaplot import figure, store, m_o_m
from cdxcore.verbose import Context
from cdxcore.pretty import PrettyObject
from quantdata.biz_calendar import DAYS_PER_YEAR
from quantdata.utils.arbfree import MkArbFree
from dataclasses import dataclass, field
import math
import numpy as np

def fit(data):
    print(data.keys()   )
    num_samples = len(data.dates)

    mkArbFree = MkArbFree()
    verbose = Context.all

    ticker = "SPX"
    for isample in range(num_samples-1,num_samples-200,-20):
        d = data.dates[isample]
        verbose.write(f"{ticker} {d}:")        

        is_valid = data.is_valid[isample,:]
        uttys = np.unique(data.tte_y[isample,:][is_valid])
        r = PrettyObject()
        r.expiries = []
        r.strikes = []
        r.bids = []
        r.asks = []

        for utty in uttys:
            mask = (data.tte_y[isample,:] == utty) & is_valid
            r.expiries.append(utty)
            r.strikes.append(data.pure_strike[isample][mask])
            r.bids.append(data.bid_pure_price[isample][mask])
            r.asks.append(data.ask_pure_price[isample][mask])
            ixs = np.argsort(r.strikes[-1])
            r.strikes[-1] = r.strikes[-1][ixs]
            r.bids[-1] = r.bids[-1][ixs]
            r.asks[-1] = r.asks[-1][ixs]
        
        af = mkArbFree( expiries=r.expiries, strikes=r.strikes, bids=r.bids, asks=r.asks,
                        verbose=verbose(1) )





fit(data)


dict_keys(['dates', 'ask_pure_delta', 'ask_pure_dk', 'ask_pure_gamma', 'ask_pure_price', 'ask_pure_theta', 'ask_pure_vega', 'ask_vol', 'bid_pure_delta', 'bid_pure_dk', 'bid_pure_gamma', 'bid_pure_price', 'bid_pure_theta', 'bid_pure_vega', 'bid_vol', 'call_ask', 'call_bid', 'df', 'fwd', 'is_am', 'is_american', 'is_valid', 'log_open_interest', 'log_volume', 'nms', 'pure_strike', 'spot_ref', 'spot_ret', 'strike', 'tte_y'])
00: SPX 2025-07-01:
01:   Generated 47 arbitrage-free expiries with extended strikes from 47 market expiries. This took 0.37s.                                
01:   Finding arbitrage-free mid prices accross the entire surface of 1014 strikes with a minimum strike distance of 0.000806... done. This took 0.66s.
00: SPX 2025-06-02:
01:   Generated 44 arbitrage-free expiries with extended strikes from 44 market expiries. This took 0.33s.
01:   Finding arbitrage-free mid prices accross the entire surface of 1021 strikes with a minimum strike distance of 0.000841... done. Thi